# Simple DMPBridge Block Comparison

This notebook keeps the evaluation simple. For each manual/reference block, it answers only three questions:

1. **Did Llama find similar text?**
2. **Did Llama give the correct label?**
3. **Was it in the correct document order?**

Input folders:

```text
data/reference_structure_blocks
data/llama_structured_blocks
```

Expected example file names:

```text
sample1_reference_blocks.json
sample1_llama_blocks.json
```


## Step 1 — Import libraries

In [1]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher
import pandas as pd

## Step 2 — Set folders

In [2]:
# Detect project root
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

reference_dir = project_root / "data" / "reference_structure_blocks"
detected_dir = project_root / "data" / "llama_structured_blocks"

output_dir = project_root / "data" / "simple_block_comparison_reports"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Reference folder:", reference_dir)
print("Detected/Llama folder:", detected_dir)
print("Output folder:", output_dir)

Project root: c:\Users\Nahid\dmpbridge
Reference folder: c:\Users\Nahid\dmpbridge\data\reference_structure_blocks
Detected/Llama folder: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks
Output folder: c:\Users\Nahid\dmpbridge\data\simple_block_comparison_reports


## Step 3 — Check files

In [3]:
print("REFERENCE FILES")
for f in sorted(reference_dir.glob("*.json")):
    print(" -", f.name)

print("\nLLAMA FILES")
for f in sorted(detected_dir.glob("*.json")):
    print(" -", f.name)

REFERENCE FILES
 - sample10_reference_blocks.json
 - sample1_reference_blocks.json
 - sample2_reference_blocks.json
 - sample3_reference_blocks.json
 - sample4_reference_blocks.json
 - sample5_reference_blocks.json
 - sample6_reference_blocks.json
 - sample7_reference_blocks.json
 - sample8_reference_blocks.json
 - sample9_reference_blocks.json

LLAMA FILES
 - sample10_llama_blocks.json
 - sample1_llama_blocks.json
 - sample2_llama_blocks.json
 - sample3_llama_blocks.json
 - sample4_llama_blocks.json
 - sample5_llama_blocks.json
 - sample6_llama_blocks.json
 - sample7_llama_blocks.json
 - sample8_llama_blocks.json
 - sample9_llama_blocks.json


## Step 4 — Helper functions

In [4]:
def normalize_text(text):
    """Clean text only for comparison. Original text is kept in the report."""
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def get_sample_id(path):
    """Extract sample ID from file names like sample1_llama_blocks.json."""
    name = path.stem.lower()
    name = name.replace("_reference_blocks", "")
    name = name.replace("_reference__blocks", "")
    name = name.replace("_llama_blocks", "")
    name = name.replace("_blocks", "")
    return name


def text_similarity(a, b):
    """Character-level text similarity from 0 to 1."""
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def load_blocks(path):
    """Load JSON blocks and keep only index, label, text, and normalized text."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    blocks = []
    for i, block in enumerate(data, start=1):
        text = block.get("text", "")
        label = block.get("label", "")
        blocks.append({
            "block_index": i,
            "label": str(label).strip(),
            "text": str(text).strip(),
            "normalized_text": normalize_text(text),
        })
    return blocks

## Step 5 — Match reference files with Llama files

In [5]:
reference_files = {get_sample_id(f): f for f in sorted(reference_dir.glob("*.json"))}
detected_files = {get_sample_id(f): f for f in sorted(detected_dir.glob("*.json"))}

common_samples = sorted(set(reference_files) & set(detected_files))

print("Matched samples:", common_samples)

for sample_id in sorted(set(reference_files) - set(detected_files)):
    print("WARNING: No Llama file found for reference sample:", sample_id)

for sample_id in sorted(set(detected_files) - set(reference_files)):
    print("WARNING: No reference file found for Llama sample:", sample_id)

Matched samples: ['sample1', 'sample10', 'sample2', 'sample3', 'sample4', 'sample5', 'sample6', 'sample7', 'sample8', 'sample9']


## Step 6 — Simple comparison settings

In [6]:
# Text threshold answers: Did Llama find similar text?
TEXT_MATCH_THRESHOLD = 0.70

# Order threshold answers: Was it in the correct document order?
# 0 means exact same block position.
# 1 means shifted by only one block, still acceptable.
ORDER_TOLERANCE = 1

print("Text match threshold:", TEXT_MATCH_THRESHOLD)
print("Order tolerance:", ORDER_TOLERANCE)

Text match threshold: 0.7
Order tolerance: 1


## Step 7 — Compare each manual block with the best Llama block

In [7]:
block_level_rows = []
summary_rows = []

for sample_id in common_samples:
    reference_blocks = load_blocks(reference_files[sample_id])
    detected_blocks = load_blocks(detected_files[sample_id])

    used_detected_indexes = set()

    for ref in reference_blocks:
        best_match = None
        best_score = 0.0

        # Find the most similar unused Llama block.
        for det in detected_blocks:
            if det["block_index"] in used_detected_indexes:
                continue

            score = text_similarity(ref["text"], det["text"])

            if score > best_score:
                best_score = score
                best_match = det

        # Question 1: Did Llama find similar text?
        found_similar_text = best_score >= TEXT_MATCH_THRESHOLD

        if found_similar_text and best_match is not None:
            used_detected_indexes.add(best_match["block_index"])

            # Question 2: Did Llama give the correct label?
            correct_label = ref["label"] == best_match["label"]

            # Question 3: Was it in the correct document order?
            order_difference = best_match["block_index"] - ref["block_index"]
            correct_order = abs(order_difference) <= ORDER_TOLERANCE

            detected_index = best_match["block_index"]
            detected_label = best_match["label"]
            detected_text = best_match["text"]
        else:
            correct_label = False
            order_difference = None
            correct_order = False
            detected_index = None
            detected_label = None
            detected_text = None

        block_level_rows.append({
            "sample_id": sample_id,
            "reference_index": ref["block_index"],
            "reference_label": ref["label"],
            "reference_text": ref["text"],
            "detected_index": detected_index,
            "detected_label": detected_label,
            "detected_text": detected_text,
            "text_similarity": round(best_score, 3),
            "found_similar_text": found_similar_text,
            "correct_label": correct_label,
            "order_difference": order_difference,
            "correct_order": correct_order,
        })

    # Summary for this sample
    sample_rows = [r for r in block_level_rows if r["sample_id"] == sample_id]
    total_reference = len(sample_rows)

    summary_rows.append({
        "sample_id": sample_id,
        "reference_blocks": total_reference,
        "llama_blocks": len(detected_blocks),
        "similar_text_found": sum(r["found_similar_text"] for r in sample_rows),
        "correct_label": sum(r["correct_label"] for r in sample_rows),
        "correct_order": sum(r["correct_order"] for r in sample_rows),
        "text_match_rate": round(sum(r["found_similar_text"] for r in sample_rows) / total_reference, 3) if total_reference else 0,
        "label_accuracy": round(sum(r["correct_label"] for r in sample_rows) / total_reference, 3) if total_reference else 0,
        "order_accuracy": round(sum(r["correct_order"] for r in sample_rows) / total_reference, 3) if total_reference else 0,
    })

block_level_df = pd.DataFrame(block_level_rows)
summary_df = pd.DataFrame(summary_rows)

summary_df

,sample_id,reference_blocks,llama_blocks,similar_text_found,correct_label,correct_order,text_match_rate,label_accuracy,order_accuracy
0,sample1,26,28,26,23,24,1.000,0.885,0.923
1,sample10,13,13,13,13,13,1.000,1.000,1.000
2,sample2,12,32,6,5,3,0.500,0.417,0.250
3,sample3,14,9,8,8,1,0.571,0.571,0.071
4,sample4,2,26,1,1,1,0.500,0.500,0.500
5,sample5,13,25,10,9,1,0.769,0.692,0.077
6,sample6,11,11,5,5,5,0.455,0.455,0.455
7,sample7,2,2,2,2,2,1.000,1.000,1.000
8,sample8,13,13,13,13,13,1.000,1.000,1.000
9,sample9,11,14,11,10,7,1.000,0.909,0.636


## Step 8 — See block-level results

In [8]:
# This table answers the three questions for every manual block.
block_level_df[[
    "sample_id",
    "reference_index",
    "reference_label",
    "detected_index",
    "detected_label",
    "text_similarity",
    "found_similar_text",
    "correct_label",
    "order_difference",
    "correct_order",
    "reference_text",
    "detected_text",
]]

,sample_id,reference_index,reference_label,detected_index,detected_label,text_similarity,found_similar_text,correct_label,order_difference,correct_order,reference_text,detected_text
0,sample1,1,document_title,1.0,document_title,1.000,True,True,0.0,True,DATA MANAGEMENT AND SHARING PLAN,DATA MANAGEMENT AND SHARING PLAN
1,sample1,2,section,2.0,section,1.000,True,True,0.0,True,Element 1: Data Type:,Element 1: Data Type:
2,sample1,3,subsection,3.0,subsection,1.000,True,True,0.0,True,A. Types and amount of scientific data expecte...,A. Types and amount of scientific data expecte...
3,sample1,4,content,4.0,content,0.999,True,True,0.0,True,This secondary data analysis project will anal...,This secondary data analysis project will anal...
4,sample1,5,subsection,5.0,section,1.000,True,False,0.0,True,B. Scientific data that will be preserved and ...,B. Scientific data that will be preserved and ...
...,...,...,...,...,...,...,...,...,...,...,...,...
112,sample9,7,content,8.0,content,0.971,True,True,1.0,True,The Program Toolkit and Activity Plans develop...,The Program Toolkit and Activity Plans develop...
113,sample9,8,section,11.0,subsection,1.000,True,False,3.0,False,"Policies and provisions for re-use, re-distrib...","Policies and provisions for re-use, re-distrib..."
114,sample9,9,content,12.0,content,0.999,True,True,3.0,False,All data will be stored in Dryad and will be a...,All data will be stored in Dryad and will be a...
115,sample9,10,section,13.0,section,1.000,True,True,3.0,False,"Archiving of Data, Samples, and Other Relevant...","Archiving of Data, Samples, and Other Relevant..."


## Step 9 — Show only problems

In [9]:
problem_df = block_level_df[
    (~block_level_df["found_similar_text"]) |
    (~block_level_df["correct_label"]) |
    (~block_level_df["correct_order"])
].copy()

problem_df[[
    "sample_id",
    "reference_index",
    "reference_label",
    "detected_index",
    "detected_label",
    "text_similarity",
    "found_similar_text",
    "correct_label",
    "order_difference",
    "correct_order",
    "reference_text",
    "detected_text",
]]

,sample_id,reference_index,reference_label,detected_index,detected_label,text_similarity,found_similar_text,correct_label,order_difference,correct_order,reference_text,detected_text
4,sample1,5,subsection,5.0,section,1.000,True,False,0.0,True,B. Scientific data that will be preserved and ...,B. Scientific data that will be preserved and ...
6,sample1,7,subsection,7.0,section,1.000,True,False,0.0,True,"C. Metadata, other relevant data, and associat...","C. Metadata, other relevant data, and associat..."
22,sample1,23,subsection,23.0,section,1.000,True,False,0.0,True,B. Whether access to scientific data will be c...,B. Whether access to scientific data will be c...
24,sample1,25,section,27.0,section,1.000,True,True,2.0,False,Element 6: Oversight of Data Management and Sh...,Element 6: Oversight of Data Management and Sh...
25,sample1,26,content,28.0,content,1.000,True,True,2.0,False,The PI and Trial & IRB Coordinator will provid...,The PI and Trial & IRB Coordinator will provid...
41,sample2,3,subsection,3.0,content,0.707,True,False,0.0,True,Data management plans should describe whether ...,Data management plans should describe whether ...
42,sample2,4,content,NaN,None,0.605,False,False,NaN,False,Roles & Responsibilities. For the proposed res...,None
43,sample2,5,section,15.0,section,1.000,True,True,10.0,False,2. Data used in publications,2. Data used in publications
44,sample2,6,subsection,NaN,None,0.234,False,False,NaN,False,Data management plans should describe how data...,None
45,sample2,7,content,NaN,None,0.691,False,False,NaN,False,Research conducted within the Center will be p...,None


## Step 10 — Simple label summary

In [10]:
# Simple accuracy by manual/reference label.
# This helps you see if title, section, subsection, or content is weakest.
label_summary_df = (
    block_level_df
    .groupby("reference_label", dropna=False)
    .agg(
        total_blocks=("reference_label", "count"),
        similar_text_found=("found_similar_text", "sum"),
        correct_label=("correct_label", "sum"),
        correct_order=("correct_order", "sum"),
    )
    .reset_index()
)

label_summary_df["text_match_rate"] = (label_summary_df["similar_text_found"] / label_summary_df["total_blocks"]).round(3)
label_summary_df["label_accuracy"] = (label_summary_df["correct_label"] / label_summary_df["total_blocks"]).round(3)
label_summary_df["order_accuracy"] = (label_summary_df["correct_order"] / label_summary_df["total_blocks"]).round(3)

label_summary_df

,reference_label,total_blocks,similar_text_found,correct_label,correct_order,text_match_rate,label_accuracy,order_accuracy
0,content,49,39,39,29,0.796,0.796,0.592
1,document_title,10,10,9,10,1.000,0.900,1.000
2,section,43,37,36,22,0.860,0.837,0.512
3,subsection,15,9,5,9,0.600,0.333,0.600


## Step 11 — Save simple reports

In [11]:
summary_path = output_dir / "summary_report.csv"
block_level_path = output_dir / "block_level_report.csv"
problem_path = output_dir / "problem_blocks_report.csv"
label_summary_path = output_dir / "label_summary_report.csv"

summary_df.to_csv(summary_path, index=False)
block_level_df.to_csv(block_level_path, index=False)
problem_df.to_csv(problem_path, index=False)
label_summary_df.to_csv(label_summary_path, index=False)

print("Saved:")
print(summary_path)
print(block_level_path)
print(problem_path)
print(label_summary_path)

Saved:
c:\Users\Nahid\dmpbridge\data\simple_block_comparison_reports\summary_report.csv
c:\Users\Nahid\dmpbridge\data\simple_block_comparison_reports\block_level_report.csv
c:\Users\Nahid\dmpbridge\data\simple_block_comparison_reports\problem_blocks_report.csv
c:\Users\Nahid\dmpbridge\data\simple_block_comparison_reports\label_summary_report.csv


## How to read the result

For each manual/reference block:

- `found_similar_text = True` means Llama detected similar text.
- `correct_label = True` means Llama used the same label as the manual/reference label.
- `correct_order = True` means the matched Llama block appeared in almost the same position in the document.

The most useful file is:

```text
problem_blocks_report.csv
```

It shows only the blocks where text, label, or order had a problem.
